# Filtrado y modelado de demanda — dataset de práctica (proxy)

**Aviso importante:** este notebook usa el dataset público de Uber en Nueva York
(2014) como *ejercicio de aprendizaje* para practicar limpieza de datos,
ingeniería de variables y entrenamiento de un modelo de demanda por hora.

**No representa datos reales de UniTransporte** (universidad en Morelos, México).
Los patrones de demanda de Manhattan (clima, transporte público, horarios,
moneda) no son equivalentes a los de un carpooling universitario en México.
Este notebook se documenta y se sube al repositorio únicamente como práctica
de la metodología, no como un modelo listo para producción.

Fuente del dataset: [`fivethirtyeight/uber-pickups-in-new-york-city`](https://www.kaggle.com/datasets/fivethirtyeight/uber-pickups-in-new-york-city) (Kaggle).


## 1. Descarga del dataset

Se descarga directo de Kaggle con `kagglehub` — así el CSV nunca se sube al repositorio, cualquiera puede reproducirlo corriendo esta celda.

In [1]:
import kagglehub

# Descarga la versión más reciente del dataset (no se guarda en el repo)
path = kagglehub.dataset_download("fivethirtyeight/uber-pickups-in-new-york-city")

print("Path to dataset files:", path)


100%|██████████| 109M/109M [00:01<00:00, 78.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/fivethirtyeight/uber-pickups-in-new-york-city/versions/2


## 2. Filtrado, agregación por hora e ingeniería de variables

Cambios respecto a la versión anterior:

- **Se completan las horas sin viajes.** Antes, si una hora no tenía ningún
  pickup en el dataset original, esa hora simplemente no aparecía como fila.
  Eso le escondía al modelo la existencia de "horas muertas". Ahora se
  reindexa a un rango horario continuo y se rellena con `demanda_viajes = 0`.
- **Se agrega codificación cíclica de la hora** (`hora_sin`, `hora_cos`).
  Codificar la hora como un entero 0–23 le hace creer al modelo que las 23:00
  y las 00:00 están lejos una de la otra, cuando en realidad son casi la
  misma hora del día. La codificación seno/coseno resuelve eso.
- **Se eliminan las columnas circulares** (`precio_estimado_usd`,
  `perfil_pasajero_probable`) del conjunto de entrenamiento. Ambas eran
  fórmulas derivadas de otras columnas del propio dataset (no datos reales),
  así que no aportan información nueva y pueden causar fuga de datos si se
  usan como *features*. Se conservan solo como columnas informativas, fuera
  del set de entrenamiento.


In [2]:
import pandas as pd
import numpy as np
import glob
import os
import holidays

# 1. Ruta de los archivos descargados por kagglehub
archivos_csv = glob.glob(os.path.join(path, "uber-raw-data-*14.csv"))

# --- CONFIGURACIÓN DE PARÁMETROS Y FILTROS ---
# Delimitación geográfica de la "zona universitaria" de práctica (área de NYU/Manhattan)
LAT_MIN, LAT_MAX = 40.7250, 40.7380
LON_MIN, LON_MAX = -74.0000, -73.9900

# Festivos de Estados Unidos / Nueva York (¡ojo!: esto es específico de NYC,
# no corresponde al calendario académico real de una universidad mexicana)
us_holidays = holidays.US(subdiv='NY')

lista_resumenes = []
print("Procesando archivos por lotes...")

# 2. Lectura y filtrado por lotes (chunking)
for archivo in archivos_csv:
    try:
        df_chunk = pd.read_csv(archivo, usecols=['Date/Time', 'Lat', 'Lon'])
    except ValueError:
        try:
            df_chunk = pd.read_csv(archivo, usecols=['Date', 'Time', 'Lat', 'Lon'])
            df_chunk['Date/Time'] = df_chunk['Date'] + ' ' + df_chunk['Time']
            df_chunk = df_chunk.drop(columns=['Date', 'Time'])
        except Exception as e:
            print(f"No se pudo leer {archivo}: {e}")
            continue

    # A) Filtrar solo viajes dentro de la zona de práctica
    df_zona = df_chunk[
        (df_chunk['Lat'] >= LAT_MIN) & (df_chunk['Lat'] <= LAT_MAX) &
        (df_chunk['Lon'] >= LON_MIN) & (df_chunk['Lon'] <= LON_MAX)
    ].copy()

    if df_zona.empty:
        continue

    # Redondear al inicio de la hora (ej: 14:23 -> 14:00)
    df_zona['Date/Time'] = pd.to_datetime(df_zona['Date/Time'])
    df_zona['fecha_hora'] = df_zona['Date/Time'].dt.floor('h')

    resumen_hora = df_zona.groupby('fecha_hora').size().reset_index(name='demanda_viajes')
    lista_resumenes.append(resumen_hora)

# 3. Consolidar
df_total = pd.concat(lista_resumenes, ignore_index=True)
df_final = df_total.groupby('fecha_hora')['demanda_viajes'].sum().reset_index()

# 4. COMPLETAR HORAS SIN VIAJES (antes faltaban por completo como filas)
rango_completo = pd.date_range(
    start=df_final['fecha_hora'].min(),
    end=df_final['fecha_hora'].max(),
    freq='h'
)
df_final = (
    df_final.set_index('fecha_hora')
    .reindex(rango_completo, fill_value=0)
    .rename_axis('fecha_hora')
    .reset_index()
)

print(f"Filas antes de completar huecos: {len(df_total['fecha_hora'].unique())}")
print(f"Filas después de completar huecos (incluye demanda=0): {len(df_final)}")

# 5. INGENIERÍA DE VARIABLES

# A) Fecha y tiempo
df_final['hora'] = df_final['fecha_hora'].dt.hour
df_final['dia_semana'] = df_final['fecha_hora'].dt.day_name()
df_final['dia_semana_num'] = df_final['fecha_hora'].dt.dayofweek  # 0=lunes
df_final['mes'] = df_final['fecha_hora'].dt.month
df_final['es_fin_de_semana'] = df_final['dia_semana_num'].isin([5, 6]).astype(int)

# B) Codificación cíclica de la hora (evita que 23h y 0h se vean "lejanas")
df_final['hora_sin'] = np.sin(2 * np.pi * df_final['hora'] / 24)
df_final['hora_cos'] = np.cos(2 * np.pi * df_final['hora'] / 24)

# C) Días festivos (de EE.UU. — solo válido para este ejercicio con datos de NYC)
df_final['es_festivo'] = df_final['fecha_hora'].dt.date.apply(lambda d: 1 if d in us_holidays else 0)

# D) Etiqueta de zona (constante, informativa, no se usa como feature)
df_final['tipo_zona'] = 'Universidad (zona de práctica, NYC)'

# --- Columnas eliminadas del set de entrenamiento por ser circulares ---
# 'precio_estimado_usd' = 8.50 + demanda*0.15  -> función de la propia demanda
# 'perfil_pasajero_probable' -> regla derivada de 'hora' y 'es_fin_de_semana'
# Ninguna de las dos aporta información nueva; se documentan aquí pero no
# se generan para evitar que alguien las use por error como feature.

# 6. Guardar CSV final (recuerda: NO subir este archivo al repo, solo el código)
df_final.to_csv('dataset_universitario_movilidad.csv', index=False)

print("\n¡Proceso completado!")
print(f"Dataset generado con {len(df_final)} registros horarios.")
df_final.head(10)


Procesando archivos por lotes...
Filas antes de completar huecos: 4382
Filas después de completar huecos (incluye demanda=0): 4391

¡Proceso completado!
Dataset generado con 4391 registros horarios.


,fecha_hora,demanda_viajes,hora,dia_semana,dia_semana_num,mes,es_fin_de_semana,hora_sin,hora_cos,es_festivo,tipo_zona
0,2014-04-01 00:00:00,5,0,Tuesday,1,4,0,0.000000,1.000000e+00,0,"Universidad (zona de práctica, NYC)"
1,2014-04-01 01:00:00,6,1,Tuesday,1,4,0,0.258819,9.659258e-01,0,"Universidad (zona de práctica, NYC)"
2,2014-04-01 02:00:00,1,2,Tuesday,1,4,0,0.500000,8.660254e-01,0,"Universidad (zona de práctica, NYC)"
3,2014-04-01 03:00:00,4,3,Tuesday,1,4,0,0.707107,7.071068e-01,0,"Universidad (zona de práctica, NYC)"
4,2014-04-01 04:00:00,6,4,Tuesday,1,4,0,0.866025,5.000000e-01,0,"Universidad (zona de práctica, NYC)"
5,2014-04-01 05:00:00,11,5,Tuesday,1,4,0,0.965926,2.588190e-01,0,"Universidad (zona de práctica, NYC)"
6,2014-04-01 06:00:00,28,6,Tuesday,1,4,0,1.000000,6.123234e-17,0,"Universidad (zona de práctica, NYC)"
7,2014-04-01 07:00:00,46,7,Tuesday,1,4,0,0.965926,-2.588190e-01,0,"Universidad (zona de práctica, NYC)"
8,2014-04-01 08:00:00,30,8,Tuesday,1,4,0,0.866025,-5.000000e-01,0,"Universidad (zona de práctica, NYC)"
9,2014-04-01 09:00:00,20,9,Tuesday,1,4,0,0.707107,-7.071068e-01,0,"Universidad (zona de práctica, NYC)"


## 3. Limpieza de nulos (imputación por reglas)

In [3]:
import numpy as np
from sklearn.impute import KNNImputer

# 1. Cargar el dataset procesado
df = pd.read_csv('dataset_universitario_movilidad.csv', parse_dates=['fecha_hora'])

target_col = 'demanda_viajes'
print(f"Variable de salida (target): {target_col}")
print("\nConteo inicial de nulos por columna:")
print(df.isnull().sum())

# Regla 1: eliminar columnas con más de 30 nulos
columnas_a_eliminar = [col for col in df.columns if df[col].isnull().sum() > 30]
if columnas_a_eliminar:
    df.drop(columns=columnas_a_eliminar, inplace=True)
    print(f"\nColumnas eliminadas por tener >30 nulos: {columnas_a_eliminar}")

# Regla 2: eliminar filas en columnas con 1 a 10 nulos
cols_1_10 = [col for col in df.columns if 1 <= df[col].isnull().sum() <= 10]
for col in cols_1_10:
    df.dropna(subset=[col], inplace=True)

# Regla 3: imputación por mediana/moda para columnas con 11 a 20 nulos
cols_11_20 = [col for col in df.columns if 11 <= df[col].isnull().sum() <= 20]
for col in cols_11_20:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col].fillna(df[col].median(), inplace=True)
    else:
        df[col].fillna(df[col].mode()[0], inplace=True)

# Regla 4: imputación con KNN para columnas con 21 a 30 nulos
cols_21_30 = [col for col in df.columns if 21 <= df[col].isnull().sum() <= 30]
if cols_21_30:
    cols_numericas = df.select_dtypes(include=[np.number]).columns
    imputer = KNNImputer(n_neighbors=5)
    df[cols_numericas] = imputer.fit_transform(df[cols_numericas])

df.to_csv('dataset_movilidad_limpio.csv', index=False)

print("\n¡Filtrado e imputación completados!")
print("\nConteo final de nulos:")
print(df.isnull().sum())


Variable de salida (target): demanda_viajes

Conteo inicial de nulos por columna:
fecha_hora          0
demanda_viajes      0
hora                0
dia_semana          0
dia_semana_num      0
mes                 0
es_fin_de_semana    0
hora_sin            0
hora_cos            0
es_festivo          0
tipo_zona           0
dtype: int64

¡Filtrado e imputación completados!

Conteo final de nulos:
fecha_hora          0
demanda_viajes      0
hora                0
dia_semana          0
dia_semana_num      0
mes                 0
es_fin_de_semana    0
hora_sin            0
hora_cos            0
es_festivo          0
tipo_zona           0
dtype: int64


## 4. Notas finales y próximos pasos

  - Propósito del prototipo: Este modelo se entrenó con datos de prueba de Nueva York únicamente para demostrar la metodología de trabajo. No se utiliza para tomar decisiones reales dentro de UniTransporte, sino como un ejemplo de cómo funcionaría la IA integrada en el proyecto.

  - Requisitos para producción: Para contar con un modelo operativo y válido, será necesario replicar este proceso utilizando datos reales generados por la app en cuanto comience a usarse (tabla viaje con fechas y horas de creación, motivos de cancelación, calendario académico real y clima local).

  - Modelo de fijación de precios: No se incluyó en este análisis porque requiere evaluar la elasticidad de la demanda (cómo reaccionan los usuarios ante variaciones de precio). Estos datos aún no existen y se irán registrando progresivamente con el historial de viajes (tanto finalizados como cancelados) para fundamentar la función principal de la IA en la plataforma.